# Notebook 04: Zero-Shot Experiments
## TSFM Industrial PdM Benchmark - ICATH Conference

**Author:** Yassire Ammouri  
**Purpose:** Run zero-shot inference for all TSFMs on all datasets  
**Models:** MOMENT, Chronos, Sundial, Time-MoE, Lag-Llama, PatchTST  
**Expected Runtime:** 2-4 hours on T4 GPU

### Step 1: Setup and Imports

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import json
import time
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
import yaml

# Add project source to path
PROJECT_DIR = Path("/content/tsfm-pdm-benchmark")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from models import get_model, list_models
from evaluation.metrics import compute_all_metrics

# Paths
PROCESSED_DIR = PROJECT_DIR / "data/processed"
RESULTS_DIR = PROJECT_DIR / "results/zero_shot"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Step 2: Configuration

In [ ]:
# Load configs
with open(PROJECT_DIR / "config/config.yaml", 'r') as f:
    config = yaml.safe_load(f)

with open(PROJECT_DIR / "config/models.yaml", 'r') as f:
    models_config = yaml.safe_load(f)

# Models to evaluate (prioritize open-source)
MODELS = ["moment", "chronos", "patchtst"]  # Add "sundial", "time_moe", "lag_llama" if installed

# Datasets to evaluate
DATASETS = ["cmapss/FD001", "cmapss/FD002", "cmapss/FD003", "cmapss/FD004"]

# Task
TASK = "forecasting"
HORIZON = config['preprocessing']['forecast_horizon']

print("Experiment Configuration:")
print(f"  Models: {MODELS}")
print(f"  Datasets: {DATASETS}")
print(f"  Task: {TASK}")
print(f"  Horizon: {HORIZON}")
print(f"  Total experiments: {len(MODELS) * len(DATASETS)}")

### Step 3: Zero-Shot Experiment Function

In [ ]:
def run_zero_shot(model_name, dataset_name, data_path, task="forecasting", horizon=96, device="cuda"):
    """Run zero-shot evaluation for a single model-dataset pair"""
    print(f"\n{'='*60}")
    print(f"Zero-Shot: {model_name} on {dataset_name}")
    print(f"{'='*60}")
    
    # Load data
    data = torch.load(data_path / "processed_data.pt")
    X_test = data['test_X']
    y_test = data['test_y']
    
    print(f"Test data shape: X={X_test.shape}, y={y_test.shape}")
    
    # Load model
    model = get_model(model_name, device=device)
    model.load_model()
    
    # Run inference
    print("Running inference...")
    start_time = time.time()
    
    results = model.zero_shot(X_test, horizon=horizon)
    predictions = results['predictions']
    
    inference_time = time.time() - start_time
    print(f"Inference time: {inference_time:.2f}s")
    
    # Handle shape mismatches
    if predictions.shape != y_test.shape:
        min_len = min(predictions.shape[1], y_test.shape[1])
        predictions = predictions[:, :min_len]
        y_test = y_test[:, :min_len]
        print(f"Shape adjusted to: {predictions.shape}")
    
    # Compute metrics
    metrics = compute_all_metrics(
        y_test.numpy() if isinstance(y_test, torch.Tensor) else y_test,
        predictions,
        task=task
    )
    
    print(f"Results: {metrics}")
    
    return {
        'model': model_name,
        'dataset': dataset_name,
        'task': task,
        'scenario': 'zero_shot',
        'metrics': metrics,
        'timestamp': datetime.now().isoformat(),
        'inference_time': inference_time,
        'predictions_shape': list(predictions.shape),
        'test_samples': len(X_test)
    }

### Step 4: Run All Zero-Shot Experiments

In [ ]:
all_results = []
errors = []

print(f"\nStarting zero-shot experiments...")
print(f"Models: {MODELS}")
print(f"Datasets: {DATASETS}")
print(f"Total: {len(MODELS) * len(DATASETS)} experiments\n")

for dataset in tqdm(DATASETS, desc="Datasets"):
    dataset_path = PROCESSED_DIR / dataset
    
    if not dataset_path.exists():
        print(f"Warning: {dataset} not found, skipping")
        continue
    
    for model_name in tqdm(MODELS, desc=f"Models ({dataset})", leave=False):
        try:
            result = run_zero_shot(
                model_name=model_name,
                dataset_name=dataset,
                data_path=dataset_path,
                task=TASK,
                horizon=HORIZON,
                device=device
            )
            all_results.append(result)
            
            # Save individual result
            safe_name = dataset.replace("/", "_")
            with open(RESULTS_DIR / f"{model_name}_{safe_name}.json", 'w') as f:
                json.dump(result, f, indent=2)
            
        except Exception as e:
            print(f"Error with {model_name} on {dataset}: {e}")
            errors.append({
                'model': model_name,
                'dataset': dataset,
                'error': str(e)
            })
    
    # Clear GPU memory after each dataset
    torch.cuda.empty_cache()

print(f"\nExperiments complete!")
print(f"Successful: {len(all_results)}")
print(f"Errors: {len(errors)}")

### Step 5: Save Results

In [ ]:
# Create results DataFrame
df_results = pd.DataFrame(all_results)

# Save to CSV
df_results.to_csv(RESULTS_DIR / "zero_shot_results.csv", index=False)
print(f"Results saved to: {RESULTS_DIR / 'zero_shot_results.csv'}")

# Save errors
if errors:
    df_errors = pd.DataFrame(errors)
    df_errors.to_csv(RESULTS_DIR / "zero_shot_errors.csv", index=False)
    print(f"Errors saved to: {RESULTS_DIR / 'zero_shot_errors.csv'}")

### Step 6: Display Results Summary

In [ ]:
print("=" * 80)
print("ZERO-SHOT RESULTS SUMMARY (MAE)")
print("=" * 80)

# Create pivot table
if len(df_results) > 0:
    pivot_data = []
    for _, row in df_results.iterrows():
        pivot_data.append({
            'model': row['model'],
            'dataset': row['dataset'],
            'mae': row['metrics'].get('mae', float('nan')),
            'rmse': row['metrics'].get('rmse', float('nan')),
            'mape': row['metrics'].get('mape', float('nan'))
        })
    
    df_pivot = pd.DataFrame(pivot_data)
    
    # Show MAE table
    mae_table = df_pivot.pivot_table(
        values='mae',
        index='model',
        columns='dataset',
        aggfunc='mean'
    )
    
    print("\nMAE (lower is better):")
    print(mae_table.to_string(float_format="%.4f"))
    
    # Best model per dataset
    print("\nBest model per dataset:")
    for dataset in df_pivot['dataset'].unique():
        ds_data = df_pivot[df_pivot['dataset'] == dataset]
        best = ds_data.loc[ds_data['mae'].idxmin()]
        print(f"  {dataset}: {best['model']} (MAE={best['mae']:.4f})")
else:
    print("No results available!")

### Step 7: Visualize Results

In [ ]:
if len(df_results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # MAE comparison
    pivot_data = []
    for _, row in df_results.iterrows():
        pivot_data.append({
            'model': row['model'],
            'dataset': row['dataset'],
            'mae': row['metrics'].get('mae', float('nan'))
        })
    
    df_plot = pd.DataFrame(pivot_data)
    
    # Bar plot
    import seaborn as sns
    sns.barplot(data=df_plot, x='model', y='mae', hue='dataset', ax=axes[0])
    axes[0].set_title('Zero-Shot MAE by Model and Dataset', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Model')
    axes[0].set_ylabel('MAE')
    axes[0].legend(title='Dataset', bbox_to_anchor=(1.02, 1), loc='upper left')
    
    # Inference time
    time_data = df_results[['model', 'dataset', 'inference_time']].copy()
    sns.barplot(data=time_data, x='model', y='inference_time', ax=axes[1])
    axes[1].set_title('Inference Time by Model', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel('Time (seconds)')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "zero_shot_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Figure saved to: {RESULTS_DIR / 'zero_shot_comparison.png'}")

### Step 8: Save to Google Drive

In [ ]:
import shutil

DRIVE_RESULTS = Path("/content/drive/MyDrive/ICATH_Results/zero_shot")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

# Copy results
for f in RESULTS_DIR.glob("*"):
    if f.is_file():
        shutil.copy(f, DRIVE_RESULTS / f.name)

print(f"Results backed up to: {DRIVE_RESULTS}")
print("\nNext: Run Notebook 05 (Few-Shot Experiments)")